# Practice Lab: Monitoring & Observability (Lesson 12.8)

Module 12 . 4 exercises . Dashboard + alerts + dispatcher + architecture


# Lesson 12.8: Monitoring & Observability

4 exercises: 2E + 1M + 1C

All exercises run **locally** (pure Python simulations).


In [ ]:
import random, numpy as np, time
random.seed(42); np.random.seed(42)


---
## Exercise 1 (Easy): 5-Metric Dashboard


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random, numpy as np
random.seed(42); np.random.seed(42)

class Dash:
    def __init__(self):
        self.lats=[]; self.errs=0; self.total=0; self.tokens=[]; self.costs=[]
        self.cache_hit=0; self.cache_tot=0; self.models={}; self.windows=[]
    def record(self,model,lat,ti,to,cost,error=False,cache=False):
        self.total+=1; self.lats.append(lat); self.tokens.append(ti+to); self.costs.append(cost)
        if error: self.errs+=1
        self.cache_tot+=1
        if cache: self.cache_hit+=1
        self.models.setdefault(model,{"c":0,"$":0,"e":0})
        self.models[model]["c"]+=1; self.models[model]["$"]+=cost
        if error: self.models[model]["e"]+=1
        if self.total%10==0:
            r=self.lats[-10:]; self.windows.append(sum(r)/len(r))
    def sparkline(self,vals):
        if not vals: return ""
        chars=" _.-~*"; mn=min(vals); rng=max(vals)-mn or 1
        return "".join(chars[min(5,int((v-mn)/rng*5))] for v in vals[-10:])

d=Dash()
for i in range(100):
    ch=random.random()<0.35
    if ch: d.record("cache",5,0,0,0,cache=True)
    elif random.random()<0.7: d.record("flash",random.gauss(800,150),random.randint(200,600),random.randint(100,400),0.0002,error=random.random()<0.03)
    elif random.random()<0.8: d.record("pro",random.gauss(2500,400),random.randint(500,1500),random.randint(200,800),0.004,error=random.random()<0.05)
    else: d.record("sonnet",random.gauss(1800,300),random.randint(400,1000),random.randint(200,600),0.005)

ls=sorted(d.lats); n=max(d.total,1)
p50=ls[len(ls)//2]; p95=ls[int(len(ls)*0.95)]; p99=ls[int(len(ls)*0.99)]
err=d.errs/n*100; at=sum(d.tokens)/n; ac=sum(d.costs)/n; cr=d.cache_hit/max(d.cache_tot,1)*100

print("5-Metric Dashboard:")
for name,val,ok,tgt in [("Latency p50",f"{p50:.0f}ms",p50<1000,"<1000"),("Latency p95",f"{p95:.0f}ms",p95<3000,"<3000"),
    ("Latency p99",f"{p99:.0f}ms",p99<5000,"<5000"),("Error rate",f"{err:.1f}%",err<5,"<5%"),
    ("Avg tokens",f"{at:.0f}",at<1500,"<1500"),("Avg cost",f"${ac:.5f}",ac<0.01,"<$0.01"),
    ("Cache hit",f"{cr:.0f}%",cr>20,">20%")]:
    print(f"  {name:<16} {val:>10} {tgt:>8} {'OK' if ok else 'WARN':>6}")
print(f"\n  Trend: [{d.sparkline(d.windows)}]")
print(f"  Models: {d.models}")


</details>


---
## Exercise 2 (Easy): Alert Test Suite


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class AP:
    def __init__(self,n,m,th,op,sev,ch): self.n=n;self.m=m;self.th=th;self.op=op;self.sev=sev;self.ch=ch
class AE:
    def __init__(self): self.policies=[]; self.fired=[]
    def add(self,p): self.policies.append(p)
    def eval(self,metrics):
        alerts=[]
        for p in self.policies:
            v=metrics.get(p.m,0); hit=(p.op=="gt" and v>p.th) or (p.op=="lt" and v<p.th)
            if hit: a={"p":p.n,"s":p.sev,"m":p.m,"v":v,"th":p.th}; alerts.append(a)
        return alerts

e=AE()
e.add(AP("High Errors","err",5,"gt","critical","pagerduty"))
e.add(AP("Slow P99","lat",5000,"gt","warning","slack"))
e.add(AP("Cost Spike","cost",50,"gt","critical","email"))
e.add(AP("Low Cache","cache",20,"lt","warning","slack"))
e.add(AP("Token Bloat","tok",2000,"gt","warning","slack"))

scenarios=[("healthy",{"err":1.2,"lat":2500,"cost":25,"cache":45,"tok":800},0),
    ("low_traffic",{"err":0,"lat":500,"cost":5,"cache":60,"tok":400},0),
    ("moderate",{"err":3,"lat":4000,"cost":40,"cache":30,"tok":1200},0),
    ("cache_ok",{"err":0.5,"lat":1000,"cost":15,"cache":55,"tok":600},0),
    ("budget_ok",{"err":2,"lat":3500,"cost":48,"cache":25,"tok":1500},0),
    ("error_spike",{"err":8.5,"lat":2000,"cost":30,"cache":40,"tok":900},1),
    ("slow",{"err":2,"lat":7500,"cost":20,"cache":35,"tok":1000},1),
    ("cost_exp",{"err":1,"lat":3000,"cost":85,"cache":50,"tok":1800},1),
    ("cache_dead",{"err":3,"lat":4500,"cost":45,"cache":8,"tok":1400},1),
    ("multi",{"err":12,"lat":8000,"cost":95,"cache":5,"tok":2500},5)]

print("Alert Test Suite:")
print(f"  {'Scenario':<15} {'Expect':>7} {'Fired':>6} {'Status':>7}")
passed=0
for name,m,exp in scenarios:
    a=e.eval(m); ok=(len(a)>=exp if exp>0 else len(a)==0); passed+=ok
    print(f"  {name:<15} {'>=' if exp else '='}{exp:>5} {len(a):>6} {'PASS' if ok else 'FAIL':>7}")
    if a:
        for al in a: print(f"    [{al['s'][:4]}] {al['p']}: {al['m']}={al['v']}")
print(f"\n  Results: {passed}/{len(scenarios)} passed")


</details>


---
## Exercise 3 (Medium): Alert Dispatcher


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import time
from datetime import datetime

class Dispatcher:
    def __init__(self,cd=0.2): self.cd=cd; self.last={}; self.sent=[]; self.supp=0
    def dispatch(self,alert):
        now=time.time(); p=alert["policy"]
        if p in self.last and now-self.last[p]<self.cd: self.supp+=1; return False
        self.last[p]=now
        icons={"critical":"!!","warning":"! ","info":"  "}
        msg=f"{icons.get(alert['severity'],'  ')} [{alert['severity'].upper()}] {alert['policy']}: {alert['metric']}={alert['value']} (>{alert['threshold']}) -> {alert['channel']}"
        self.sent.append(msg); return True

d=Dispatcher(0.2)
print("Alert Dispatcher (20 rapid alerts):")
alerts=[{"policy":"High Errors","severity":"critical","metric":"error_rate","value":8.5,"threshold":5,"channel":"pagerduty"},
    {"policy":"Slow P99","severity":"warning","metric":"latency","value":6200,"threshold":5000,"channel":"slack"},
    {"policy":"Cost Spike","severity":"critical","metric":"daily_cost","value":75,"threshold":50,"channel":"email"},
    {"policy":"Low Cache","severity":"warning","metric":"cache_hit","value":12,"threshold":20,"channel":"slack"}]
for i in range(20):
    sent=d.dispatch(alerts[i%4])
    if sent: print(f"  [{i+1:>2}] SENT: {d.sent[-1]}")
    if i==7: time.sleep(0.25)
print(f"\n  Sent: {len(d.sent)} | Suppressed: {d.supp} ({d.supp/20*100:.0f}%)")


</details>


---
## Exercise 4 (Challenge): Cloud Monitoring Arch


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
print("Cloud Monitoring Architecture:")
print("  App -> stdout JSON -> Cloud Logging")
print("  App -> /metrics -> Cloud Monitoring")
print("  App -> OpenTelemetry -> Cloud Trace")
print("  App -> Langfuse (self-hosted, LLM traces)")
print("\n  8 Dashboard Charts:")
for c in ["Latency p50/p95/p99 (line)","Error Rate (line)","Requests/min (line)","Token Usage (area)",
    "Cost/Team (bar)","Cost/Model (pie)","Cache Hit% (gauge)","Instances (line)"]:
    print(f"    - {c}")
print("\n  5 Alert Policies:")
for n,c,s in [("High Errors","err>5%","critical"),("Slow P99","p99>5s","warning"),
    ("Cost Spike","$>50/day","critical"),("Low Cache","cache<20%","warning"),("Instance Spike","inst>20","warning")]:
    print(f"    [{s:>8}] {n}: {c}")
costs=[("Cloud Logging",0,25),("Cloud Monitoring",0,10),("Cloud Trace",0,10),("Langfuse",5,15),("Alerts",0,5)]
tl=sum(l for _,l,_ in costs); th=sum(h for _,_,h in costs)
print(f"\n  Monthly cost: ${tl}-${th} (generous free tiers)")


</details>
